In [1]:
!pip install pgmpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 84.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu1

In [ ]:
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork

# Import the necessary estimators based on your provided import structure
from pgmpy.estimators.HillClimbSearch import HillClimbSearch
from pgmpy.estimators.StructureScore import BIC,AIC   # Using BIC as the scoring method
from pgmpy.estimators.MLE import MaximumLikelihoodEstimator

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Define the columns (features) as provided
cols = [
    "loan_amnt_disc",
    "int_rate_disc",
    "installment_disc",
    "pub_rec_disc",
    "revol_bal_disc",
    "revol_util_disc",
    "total_acc_disc",
    "annual_inc_disc",
    "delinq_2yrs_disc",
    "inq_last_6mths_disc",
    "open_acc_disc",
    "gdp_growth_t1_disc",
    "unemployment_rate_t1_disc",
    "fedfunds_t1_disc",
    "inflation_t1_disc",
    "housing_price_t1_disc",
    "grade",
    "sub_grade",
    "emp_length",
    "home_ownership",
    "purpose",
    "fico_bucket_pct",
    "default_indicator",
    "age_group",
    "issue_quarter",
]


# Load your pre-discretized data
data = pd.read_csv('rebalanced_age_data_discretized.csv')

# Filter the data to include only the relevant columns
data = data[cols]
data_no_na_default = data.dropna(subset=["default_indicator"])
data = data_no_na_default

# Split the data into training and test sets.
train_data, test_data = train_test_split(data, test_size=0.025, random_state=454, stratify=data["default_indicator"])

print("Total data size:", len(data))

# ----------------------------
# Structure Learning Section
# ----------------------------

# Initialize the AIC scoring method with the training data.
aic = AIC(train_data)

# Create a Hill Climb Search object (scoring method is passed later).
hc = HillClimbSearch(train_data)

# Estimate the best model structure using AIC.
best_model = hc.estimate(scoring_method=aic)

# Print the learned structure edges.
print("Learned structure edges:", best_model.edges())

# ----------------------------
# Parameter Learning Section
# ----------------------------

# Create a new BayesianNetwork model with the learned structure.
model = DiscreteBayesianNetwork(best_model.edges())

# Fit the network on the training data using Maximum Likelihood Estimation.
model.fit(train_data, estimator=MaximumLikelihoodEstimator)

# Print the learned CPDs for each node.
print("Learned CPDs:")
for cpd in model.get_cpds():
    print(cpd)

# ----------------------------
# Inference and Evaluation Section
# ----------------------------

# Prepare test data by removing the target column for inference.
test_features = test_data.drop(columns=["default_indicator"])

predictions_list = []
batch_size = 1000 # Adjust based on available memory
for i in range(0, len(test_features), batch_size):
    batch = test_features.iloc[i:i+batch_size]
    predictions_batch = model.predict(batch)
    predictions_list.append(predictions_batch)

# Combine predictions from all batches
predictions = pd.concat(predictions_list)
# Extract predictions and ground truth for evaluation.
y_pred = predictions["default_indicator"]
y_true = test_data["default_indicator"]

Total data size: 183893


  0%|          | 0/1000000 [00:00<?, ?it/s]

Learned structure edges: [('int_rate_disc', 'loan_amnt_disc'), ('int_rate_disc', 'revol_util_disc'), ('int_rate_disc', 'inq_last_6mths_disc'), ('int_rate_disc', 'total_acc_disc'), ('int_rate_disc', 'installment_disc'), ('installment_disc', 'loan_amnt_disc'), ('revol_util_disc', 'revol_bal_disc'), ('total_acc_disc', 'open_acc_disc'), ('total_acc_disc', 'inq_last_6mths_disc'), ('total_acc_disc', 'delinq_2yrs_disc'), ('total_acc_disc', 'revol_bal_disc'), ('annual_inc_disc', 'emp_length'), ('inq_last_6mths_disc', 'revol_util_disc'), ('open_acc_disc', 'revol_bal_disc'), ('gdp_growth_t1_disc', 'housing_price_t1_disc'), ('gdp_growth_t1_disc', 'fedfunds_t1_disc'), ('unemployment_rate_t1_disc', 'grade'), ('unemployment_rate_t1_disc', 'inq_last_6mths_disc'), ('unemployment_rate_t1_disc', 'open_acc_disc'), ('unemployment_rate_t1_disc', 'total_acc_disc'), ('inflation_t1_disc', 'issue_quarter'), ('inflation_t1_disc', 'gdp_growth_t1_disc'), ('inflation_t1_disc', 'fedfunds_t1_disc'), ('inflation_t1_d

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/598 [00:00<?, ?it/s]

In [ ]:
import numpy as np
import pandas as pd
from pgmpy.inference import VariableElimination
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score

# -----------------------------------------------------------
# Function to generate predictions given a threshold
# -----------------------------------------------------------
def generate_predictions(inference, test_data, threshold):
    predictions = []
    # Loop over test instances; for each, drop the target to create evidence.
    for idx, row in test_data.drop(columns=["default_indicator"]).iterrows():
        evidence = row.to_dict()
        posterior = inference.query(variables=['default_indicator'], evidence=evidence)
        # Assume that index 1 corresponds to the positive state.
        pred = 1 if posterior.values[1] > threshold else 0
        predictions.append(pred)
    return pd.Series(predictions, index=test_data.index)

# -----------------------------------------------------------
# Initial Setup: Inference Object and Data Preparation
# -----------------------------------------------------------
# Create an inference object.
inference = VariableElimination(model)

# Ensure no missing values in test_data.
test_data = test_data.dropna()

# Ground truth labels.
y_true = test_data["default_indicator"]

# -----------------------------------------------------------
# Search for the Optimal Threshold Maximizing F1 Score for Class 1
# -----------------------------------------------------------
threshold_candidates = np.linspace(0.0, 1.0, 101)  # 101 candidate thresholds.
best_threshold = None
best_f1 = -np.inf  # Start with a very low value.

for threshold in threshold_candidates:
    predictions = generate_predictions(inference, test_data, threshold)

    # Calculate F1 score for class 1 using the binary setting.
    current_f1 = f1_score(y_true, predictions, pos_label=1, zero_division=0)

    # Update if this threshold gives a better F1 score.
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_threshold = threshold

print(f"Optimal threshold for maximizing F1 score for class 1: {best_threshold:.2f}")
print(f"Maximum F1 score for class 1 achieved: {best_f1:.2f}")

# -----------------------------------------------------------
# Evaluate with the Selected Threshold
# -----------------------------------------------------------
optimal_predictions = generate_predictions(inference, test_data, best_threshold)
accuracy = accuracy_score(y_true, optimal_predictions)
print("\nClassification Accuracy:", accuracy)
print("\nClassification Report:")
print(classification_report(y_true, optimal_predictions))
print("Confusion Matrix:")
print(confusion_matrix(y_true, optimal_predictions))

Optimal threshold for maximizing F1 score for class 1: 0.16
Maximum F1 score for class 1 achieved: 0.32

Classification Accuracy: 0.576876659425537

Classification Report:
              precision    recall  f1-score   support

         0.0       0.89      0.57      0.69      3481
         1.0       0.22      0.63      0.32       662

    accuracy                           0.58      4143
   macro avg       0.55      0.60      0.51      4143
weighted avg       0.78      0.58      0.63      4143

Confusion Matrix:
[[1976 1505]
 [ 248  414]]


In [ ]:
count_nan_y_true = y_true.isna().sum()
df = pd.DataFrame({
    'y_true': y_true,
    'y_pred': y_pred
})
print("\nDataFrame with predictions:\n", df)
print("\nCount of NaN values in y_pred:", y_pred.isna().sum())
print("\nCount of NaN values in y_true:", count_nan_y_true)

df_no_na = df.dropna()
accuracy = accuracy_score(df_no_na['y_true'], df_no_na['y_pred'])
print("\nClassification Accuracy:", accuracy)
print("\nClassification Report:\n", classification_report(df_no_na['y_true'], df_no_na['y_pred']))
print("Confusion Matrix:\n", confusion_matrix(df_no_na['y_true'], df_no_na['y_pred']))


DataFrame with predictions:
         y_true  y_pred
0          0.0     0.0
79         0.0     0.0
205        NaN     0.0
229        0.0     0.0
230        0.0     0.0
...        ...     ...
183749     NaN     0.0
183760     NaN     0.0
183807     NaN     0.0
183827     NaN     0.0
183879     NaN     0.0

[4598 rows x 2 columns]

Count of NaN values in y_pred: 0

Count of NaN values in y_true: 0

Classification Accuracy: 0.8409365194303645

Classification Report:
               precision    recall  f1-score   support

         0.0       0.84      1.00      0.91      3481
         1.0       0.71      0.01      0.01       662

    accuracy                           0.84      4143
   macro avg       0.78      0.50      0.46      4143
weighted avg       0.82      0.84      0.77      4143

Confusion Matrix:
 [[3479    2]
 [ 657    5]]


In [ ]:
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork

# Import the necessary estimators based on your provided import structure
from pgmpy.estimators.HillClimbSearch import HillClimbSearch
from pgmpy.estimators.StructureScore import BIC,AIC,BDeu  # Using BIC as the scoring method
from pgmpy.estimators.MLE import MaximumLikelihoodEstimator

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Define the columns (features) as provided
cols = [
    "loan_amnt_disc",
    "int_rate_disc",
    "installment_disc",
    "pub_rec_disc",
    "revol_bal_disc",
    "revol_util_disc",
    "total_acc_disc",
    "annual_inc_disc",
    "delinq_2yrs_disc",
    "inq_last_6mths_disc",
    "open_acc_disc",
    "gdp_growth_t1_disc",
    "unemployment_rate_t1_disc",
    "fedfunds_t1_disc",
    "inflation_t1_disc",
    "housing_price_t1_disc",
    "grade",
    "sub_grade",
    "emp_length",
    "home_ownership",
    "purpose",
    "fico_bucket_pct",
    "default_indicator",
    "age_group",
    "issue_quarter",
]


# Load your pre-discretized data
data = pd.read_csv('rebalanced_age_data_discretized.csv')

# Filter the data to include only the relevant columns
data = data[cols]
data_no_na_default = data.dropna(subset=["default_indicator"])
data = data_no_na_default

# Split the data into training and test sets.
train_data, test_data = train_test_split(data, test_size=0.025, random_state=454, stratify=data["default_indicator"])

print("Total data size:", len(data))

# ----------------------------
# Structure Learning Section
# ----------------------------

# Initialize the AIC scoring method with the training data.
aic = BDeu(train_data)

# Create a Hill Climb Search object (scoring method is passed later).
hc = HillClimbSearch(train_data)

# Estimate the best model structure using AIC.
best_model = hc.estimate(scoring_method=aic)

# Print the learned structure edges.
print("Learned structure edges:", best_model.edges())

# ----------------------------
# Parameter Learning Section
# ----------------------------

# Create a new BayesianNetwork model with the learned structure.
model = DiscreteBayesianNetwork(best_model.edges())

# Fit the network on the training data using Maximum Likelihood Estimation.
model.fit(train_data, estimator=MaximumLikelihoodEstimator)

# Print the learned CPDs for each node.
print("Learned CPDs:")
for cpd in model.get_cpds():
    print(cpd)

# ----------------------------
# Inference and Evaluation Section
# ----------------------------

# Prepare test data by removing the target column for inference.
test_features = test_data.drop(columns=["default_indicator"])

predictions_list = []
batch_size = 1000 # Adjust based on available memory
for i in range(0, len(test_features), batch_size):
    batch = test_features.iloc[i:i+batch_size]
    predictions_batch = model.predict(batch)
    predictions_list.append(predictions_batch)

# Combine predictions from all batches
predictions = pd.concat(predictions_list)
# Extract predictions and ground truth for evaluation.
y_pred = predictions["default_indicator"]
y_true = test_data["default_indicator"]

Total data size: 183893


  0%|          | 0/1000000 [00:00<?, ?it/s]

Learned structure edges: [('int_rate_disc', 'loan_amnt_disc'), ('int_rate_disc', 'revol_util_disc'), ('int_rate_disc', 'inq_last_6mths_disc'), ('int_rate_disc', 'total_acc_disc'), ('installment_disc', 'loan_amnt_disc'), ('revol_util_disc', 'revol_bal_disc'), ('revol_util_disc', 'inq_last_6mths_disc'), ('total_acc_disc', 'open_acc_disc'), ('total_acc_disc', 'delinq_2yrs_disc'), ('annual_inc_disc', 'age_group'), ('annual_inc_disc', 'emp_length'), ('annual_inc_disc', 'grade'), ('annual_inc_disc', 'default_indicator'), ('inq_last_6mths_disc', 'total_acc_disc'), ('open_acc_disc', 'revol_bal_disc'), ('gdp_growth_t1_disc', 'housing_price_t1_disc'), ('gdp_growth_t1_disc', 'fedfunds_t1_disc'), ('gdp_growth_t1_disc', 'unemployment_rate_t1_disc'), ('unemployment_rate_t1_disc', 'fedfunds_t1_disc'), ('fedfunds_t1_disc', 'pub_rec_disc'), ('inflation_t1_disc', 'issue_quarter'), ('inflation_t1_disc', 'gdp_growth_t1_disc'), ('inflation_t1_disc', 'fedfunds_t1_disc'), ('inflation_t1_disc', 'housing_price

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/598 [00:00<?, ?it/s]

In [ ]:
count_nan_y_true = y_true.isna().sum()
df = pd.DataFrame({
    'y_true': y_true,
    'y_pred': y_pred
})
print("\nDataFrame with predictions:\n", df)
print("\nCount of NaN values in y_pred:", y_pred.isna().sum())
print("\nCount of NaN values in y_true:", count_nan_y_true)

df_no_na = df.dropna()
accuracy = accuracy_score(df_no_na['y_true'], df_no_na['y_pred'])
print("\nClassification Accuracy:", accuracy)
print("\nClassification Report:\n", classification_report(df_no_na['y_true'], df_no_na['y_pred']))
print("Confusion Matrix:\n", confusion_matrix(df_no_na['y_true'], df_no_na['y_pred']))


DataFrame with predictions:
         y_true  y_pred
0          0.0     0.0
79         0.0     0.0
205        0.0     0.0
229        0.0     0.0
230        0.0     0.0
...        ...     ...
183749     0.0     0.0
183760     0.0     0.0
183807     0.0     0.0
183827     1.0     0.0
183879     0.0     0.0

[4598 rows x 2 columns]

Count of NaN values in y_pred: 0

Count of NaN values in y_true: 0

Classification Accuracy: 0.8314484558503697

Classification Report:
               precision    recall  f1-score   support

         0.0       0.83      1.00      0.91      3824
         1.0       0.00      0.00      0.00       774

    accuracy                           0.83      4598
   macro avg       0.42      0.50      0.45      4598
weighted avg       0.69      0.83      0.76      4598

Confusion Matrix:
 [[3823    1]
 [ 774    0]]


In [ ]:
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork

# Import the necessary estimators based on your provided import structure
from pgmpy.estimators.HillClimbSearch import HillClimbSearch
from pgmpy.estimators.StructureScore import BIC,AIC,BDeu,BDs  # Using BIC as the scoring method
from pgmpy.estimators.MLE import MaximumLikelihoodEstimator

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Define the columns (features) as provided
cols = [
    "age_group",
    "emp_length",
    "fico_bucket_pct",
    "purpose",
    "home_ownership",
    "issue_quarter",
    "gdp_growth_t1_disc",
    "unemployment_rate_t1_disc",
    "fedfunds_t1_disc",
    "inflation_t1_disc",
    "housing_price_t1_disc",
    "annual_inc_disc",
    "open_acc_disc",
    "pub_rec_disc",
    "revol_bal_disc",
    "revol_util_disc",
    "delinq_2yrs_disc",
    "inq_last_6mths_disc",
    "grade",
    "sub_grade",
    "total_acc_disc",
    "loan_amnt_disc",
    "int_rate_disc",
    "installment_disc",
    "default_indicator",
]


# Load your pre-discretized data
data = pd.read_csv('rebalanced_age_data_discretized.csv')

# Filter the data to include only the relevant columns
data = data[cols]
data_no_na_default = data.dropna(subset=["default_indicator"])
data = data_no_na_default

# Split the data into training and test sets.
train_data, test_data = train_test_split(data, test_size=0.025, random_state=454, stratify=data["default_indicator"])

print("Total data size:", len(data))

# ----------------------------
# Structure Learning Section
# ----------------------------

# Initialize the AIC scoring method with the training data.
aic = BDs(train_data)

# Create a Hill Climb Search object (scoring method is passed later).
hc = HillClimbSearch(train_data)

# Estimate the best model structure using AIC.
best_model = hc.estimate(scoring_method=aic)

# Print the learned structure edges.
print("Learned structure edges:", best_model.edges())

# ----------------------------
# Parameter Learning Section
# ----------------------------

# Create a new BayesianNetwork model with the learned structure.
model = DiscreteBayesianNetwork(best_model.edges())

# Fit the network on the training data using Maximum Likelihood Estimation.
model.fit(train_data, estimator=MaximumLikelihoodEstimator)

# Print the learned CPDs for each node.
print("Learned CPDs:")
for cpd in model.get_cpds():
    print(cpd)

# ----------------------------
# Inference and Evaluation Section
# ----------------------------

# Prepare test data by removing the target column for inference.
test_features = test_data.drop(columns=["default_indicator"])

predictions_list = []
batch_size = 1000 # Adjust based on available memory
for i in range(0, len(test_features), batch_size):
    batch = test_features.iloc[i:i+batch_size]
    predictions_batch = model.predict(batch)
    predictions_list.append(predictions_batch)

# Combine predictions from all batches
predictions = pd.concat(predictions_list)
# Extract predictions and ground truth for evaluation.
y_pred = predictions["default_indicator"]
y_true = test_data["default_indicator"]

Total data size: 183893


  0%|          | 0/1000000 [00:00<?, ?it/s]

Learned structure edges: [('emp_length', 'issue_quarter'), ('emp_length', 'age_group'), ('emp_length', 'purpose'), ('emp_length', 'grade'), ('emp_length', 'sub_grade'), ('emp_length', 'home_ownership'), ('emp_length', 'installment_disc'), ('emp_length', 'inflation_t1_disc'), ('emp_length', 'int_rate_disc'), ('emp_length', 'total_acc_disc'), ('emp_length', 'fico_bucket_pct'), ('emp_length', 'revol_util_disc'), ('emp_length', 'open_acc_disc'), ('emp_length', 'default_indicator'), ('emp_length', 'inq_last_6mths_disc'), ('emp_length', 'revol_bal_disc'), ('emp_length', 'loan_amnt_disc'), ('emp_length', 'delinq_2yrs_disc'), ('emp_length', 'pub_rec_disc'), ('fico_bucket_pct', 'inflation_t1_disc'), ('purpose', 'home_ownership'), ('home_ownership', 'age_group'), ('issue_quarter', 'housing_price_t1_disc'), ('issue_quarter', 'unemployment_rate_t1_disc'), ('issue_quarter', 'gdp_growth_t1_disc'), ('issue_quarter', 'fedfunds_t1_disc'), ('issue_quarter', 'int_rate_disc'), ('gdp_growth_t1_disc', 'hous

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/598 [00:00<?, ?it/s]

In [ ]:
count_nan_y_true = y_true.isna().sum()
df = pd.DataFrame({
    'y_true': y_true,
    'y_pred': y_pred
})
print("\nDataFrame with predictions:\n", df)
print("\nCount of NaN values in y_pred:", y_pred.isna().sum())
print("\nCount of NaN values in y_true:", count_nan_y_true)

df_no_na = df.dropna()
accuracy = accuracy_score(df_no_na['y_true'], df_no_na['y_pred'])
print("\nClassification Accuracy:", accuracy)
print("\nClassification Report:\n", classification_report(df_no_na['y_true'], df_no_na['y_pred']))
print("Confusion Matrix:\n", confusion_matrix(df_no_na['y_true'], df_no_na['y_pred']))


DataFrame with predictions:
         y_true  y_pred
0          0.0     0.0
79         0.0     0.0
205        0.0     0.0
229        0.0     0.0
230        0.0     0.0
...        ...     ...
183749     0.0     0.0
183760     0.0     0.0
183807     0.0     0.0
183827     1.0     0.0
183879     0.0     0.0

[4598 rows x 2 columns]

Count of NaN values in y_pred: 0

Count of NaN values in y_true: 0

Classification Accuracy: 0.8314484558503697

Classification Report:
               precision    recall  f1-score   support

         0.0       0.83      1.00      0.91      3824
         1.0       0.00      0.00      0.00       774

    accuracy                           0.83      4598
   macro avg       0.42      0.50      0.45      4598
weighted avg       0.69      0.83      0.76      4598

Confusion Matrix:
 [[3823    1]
 [ 774    0]]


In [ ]:
from pgmpy.inference import VariableElimination
# -----------------------------------------------------------
# Inference & Prediction with a 0.5 Threshold for Default
# -----------------------------------------------------------
# Create an inference object.
inference = VariableElimination(model)

# Set the prediction threshold for default_indicator.
threshold = 0.2

# Prepare a list for storing predictions.
predictions = []

test_data = test_data.dropna()

# Loop over test instances. For each instance, drop the target value to create evidence.
for idx, row in test_data.drop(columns=["default_indicator"]).iterrows():
    evidence = row.to_dict()
    # Query the posterior distribution for default_indicator.
    posterior = inference.query(variables=['default_indicator'], evidence=evidence)

    # Retrieve the probability for each state.
    # Here we assume that the states are ordered so that index 1 corresponds to the "default" state.
    # (Adjust the index if your state ordering is different.)
    prob_default = posterior.values[1]

    # Apply the threshold: predict default (1) if prob_default > threshold; otherwise predict 0.
    pred = 1 if prob_default > threshold else 0
    predictions.append(pred)

# Convert predictions to a Pandas Series.
predictions = pd.Series(predictions, index=test_data.index)

# Extract the ground truth from test data.
y_true = test_data["default_indicator"]

# Evaluate the predictions.
accuracy = accuracy_score(y_true, predictions)
print("\nClassification Accuracy:", accuracy)
print("\nClassification Report:")
print(classification_report(y_true, predictions))
print("Confusion Matrix:")
print(confusion_matrix(y_true, predictions))

/usr/local/lib/python3.11/dist-packages/pgmpy/factors/discrete/DiscreteFactor.py:489: RuntimeWarning: invalid value encountered in divide
  phi.values = phi.values / (phi.values.sum())



Classification Accuracy: 0.7052860246198407

Classification Report:
              precision    recall  f1-score   support

         0.0       0.86      0.77      0.82      3481
         1.0       0.22      0.34      0.27       662

    accuracy                           0.71      4143
   macro avg       0.54      0.56      0.54      4143
weighted avg       0.76      0.71      0.73      4143

Confusion Matrix:
[[2694  787]
 [ 434  228]]


In [ ]:
import numpy as np
import pandas as pd
from pgmpy.inference import VariableElimination
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score

# -----------------------------------------------------------
# Function to generate predictions given a threshold
# -----------------------------------------------------------
def generate_predictions(inference, test_data, threshold):
    predictions = []
    # Loop over test instances; for each, drop the target to create evidence.
    for idx, row in test_data.drop(columns=["default_indicator"]).iterrows():
        evidence = row.to_dict()
        posterior = inference.query(variables=['default_indicator'], evidence=evidence)
        # Assume that index 1 corresponds to the positive state.
        pred = 1 if posterior.values[1] > threshold else 0
        predictions.append(pred)
    return pd.Series(predictions, index=test_data.index)

# -----------------------------------------------------------
# Initial Setup: Inference Object and Data Preparation
# -----------------------------------------------------------
# Create an inference object.
inference = VariableElimination(model)

# Ensure no missing values in test_data.
test_data = test_data.dropna()

# Ground truth labels.
y_true = test_data["default_indicator"]

# -----------------------------------------------------------
# Search for the Optimal Threshold Maximizing F1 Score for Class 1
# -----------------------------------------------------------
threshold_candidates = np.linspace(0.0, 1.0, 101)  # 101 candidate thresholds.
best_threshold = None
best_f1 = -np.inf  # Start with a very low value.

for threshold in threshold_candidates:
    predictions = generate_predictions(inference, test_data, threshold)

    # Calculate F1 score for class 1 using the binary setting.
    current_f1 = f1_score(y_true, predictions, pos_label=1, zero_division=0)

    # Update if this threshold gives a better F1 score.
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_threshold = threshold

print(f"Optimal threshold for maximizing F1 score for class 1: {best_threshold:.2f}")
print(f"Maximum F1 score for class 1 achieved: {best_f1:.2f}")

# -----------------------------------------------------------
# Evaluate with the Selected Threshold
# -----------------------------------------------------------
optimal_predictions = generate_predictions(inference, test_data, best_threshold)
accuracy = accuracy_score(y_true, optimal_predictions)
print("\nClassification Accuracy:", accuracy)
print("\nClassification Report:")
print(classification_report(y_true, optimal_predictions))
print("Confusion Matrix:")
print(confusion_matrix(y_true, optimal_predictions))

/usr/local/lib/python3.11/dist-packages/pgmpy/factors/discrete/DiscreteFactor.py:489: RuntimeWarning: invalid value encountered in divide
  phi.values = phi.values / (phi.values.sum())
/usr/local/lib/python3.11/dist-packages/pgmpy/factors/discrete/DiscreteFactor.py:489: RuntimeWarning: invalid value encountered in divide
  phi.values = phi.values / (phi.values.sum())
/usr/local/lib/python3.11/dist-packages/pgmpy/factors/discrete/DiscreteFactor.py:489: RuntimeWarning: invalid value encountered in divide
  phi.values = phi.values / (phi.values.sum())
/usr/local/lib/python3.11/dist-packages/pgmpy/factors/discrete/DiscreteFactor.py:489: RuntimeWarning: invalid value encountered in divide
  phi.values = phi.values / (phi.values.sum())
/usr/local/lib/python3.11/dist-packages/pgmpy/factors/discrete/DiscreteFactor.py:489: RuntimeWarning: invalid value encountered in divide
  phi.values = phi.values / (phi.values.sum())
/usr/local/lib/python3.11/dist-packages/pgmpy/factors/discrete/DiscreteFact

Optimal threshold for maximizing F1 score for class 1: 0.14
Maximum F1 score for class 1 achieved: 0.32


/usr/local/lib/python3.11/dist-packages/pgmpy/factors/discrete/DiscreteFactor.py:489: RuntimeWarning: invalid value encountered in divide
  phi.values = phi.values / (phi.values.sum())



Classification Accuracy: 0.5307748008689356

Classification Report:
              precision    recall  f1-score   support

         0.0       0.89      0.50      0.64      3481
         1.0       0.21      0.68      0.32       662

    accuracy                           0.53      4143
   macro avg       0.55      0.59      0.48      4143
weighted avg       0.78      0.53      0.59      4143

Confusion Matrix:
[[1749 1732]
 [ 212  450]]


In [ ]:
df = pd.read_csv('rebalanced_age_data_discretized.csv')
cols = [
    "age_group",
    "emp_length",
    "fico_bucket_pct",
    "purpose",
    "home_ownership",
    "issue_quarter",
    "gdp_growth_t1_disc",
    "unemployment_rate_t1_disc",
    "fedfunds_t1_disc",
    "inflation_t1_disc",
    "housing_price_t1_disc",
    "annual_inc_disc",
    "open_acc_disc",
    "pub_rec_disc",
    "revol_bal_disc",
    "revol_util_disc",
    "delinq_2yrs_disc",
    "inq_last_6mths_disc",
    "grade",
    "sub_grade",
    "total_acc_disc",
    "loan_amnt_disc",
    "int_rate_disc",
    "installment_disc",
    "default_indicator",
]

df = df[cols]

df.head()

,age_group,emp_length,fico_bucket_pct,purpose,home_ownership,issue_quarter,gdp_growth_t1_disc,unemployment_rate_t1_disc,fedfunds_t1_disc,inflation_t1_disc,...,revol_util_disc,delinq_2yrs_disc,inq_last_6mths_disc,grade,sub_grade,total_acc_disc,loan_amnt_disc,int_rate_disc,installment_disc,default_indicator
0,25-33,10+ years,Deep Subprime,debt_consolidation,MORTGAGE,2015Q4,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,C,C4,0.0,1.0,1.0,1.0,0.0
1,34-40,3 years,Near Prime,other,MORTGAGE,2015Q4,1.0,0.0,0.0,0.0,...,1.0,0.0,0.0,C,C2,1.0,0.0,1.0,0.0,0.0
2,18-24,2 years,Deep Subprime,credit_card,RENT,2015Q4,1.0,0.0,0.0,0.0,...,1.0,0.0,0.0,C,C3,0.0,2.0,1.0,2.0,0.0
3,18-24,5 years,Prime,debt_consolidation,RENT,2015Q4,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,C,C5,0.0,2.0,1.0,2.0,1.0
4,25-33,10+ years,Deep Subprime,other,MORTGAGE,2015Q4,1.0,0.0,0.0,0.0,...,0.0,1.0,1.0,D,D3,1.0,1.0,2.0,1.0,0.0


In [ ]:
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork

# Import required estimators and inference methods
from pgmpy.estimators.HillClimbSearch import HillClimbSearch
from pgmpy.estimators.StructureScore import K2
from pgmpy.estimators.MLE import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Define your feature columns for the credit risk model.
cols = [
    "age_group",
    "emp_length",
    "fico_bucket_pct",
    "purpose",
    "home_ownership",
    "issue_quarter",
    "gdp_growth_t1_disc",
    "unemployment_rate_t1_disc",
    "fedfunds_t1_disc",
    "inflation_t1_disc",
    "housing_price_t1_disc",
    "annual_inc_disc",
    "open_acc_disc",
    "pub_rec_disc",
    "revol_bal_disc",
    "revol_util_disc",
    "delinq_2yrs_disc",
    "inq_last_6mths_disc",
    "grade",
    "sub_grade",
    "total_acc_disc",
    "loan_amnt_disc",
    "int_rate_disc",
    "installment_disc",
    "default_indicator",
]

# Load your pre-discretized data.
data = pd.read_csv('rebalanced_age_data_discretized.csv')
data = data[cols]
data_no_na_default = data.dropna(subset=["default_indicator"])
data = data_no_na_default

# Split the data into training and testing sets (here, 5% test split).
train_data, test_data = train_test_split(data, test_size=0.025, random_state=42, stratify=data["default_indicator"])
print("Total data size:", len(data))

# -----------------------------------------------------------
# Structure Learning using the K2 Score with a Specified Ordering
# -----------------------------------------------------------
# For K2, the ordering determines which nodes can be parents of which.
# Here we want all the predictors to potentially influence default_indicator.
ordering = [
    "age_group",
    "emp_length",
    "fico_bucket_pct",
    "purpose",
    "home_ownership",
    "issue_quarter",
    "gdp_growth_t1_disc",
    "unemployment_rate_t1_disc",
    "fedfunds_t1_disc",
    "inflation_t1_disc",
    "housing_price_t1_disc",
    "annual_inc_disc",
    "open_acc_disc",
    "pub_rec_disc",
    "revol_bal_disc",
    "revol_util_disc",
    "delinq_2yrs_disc",
    "inq_last_6mths_disc",
    "grade",
    "sub_grade",
    "total_acc_disc",
    "loan_amnt_disc",
    "int_rate_disc",
    "installment_disc",
    "default_indicator",
]

# Initialize the K2 score using the training data and specified ordering.
k2 = K2(train_data)

# Use Hill Climb Search for structure learning.
hc = HillClimbSearch(train_data)
best_model = hc.estimate(scoring_method=k2)

print("Learned structure edges using K2 Score:")
print(best_model.edges())

# -----------------------------------------------------------
# Parameter Learning
# -----------------------------------------------------------
# Create the Bayesian Network model with the learned structure.
model = DiscreteBayesianNetwork(best_model.edges())

# Learn the CPDs using Maximum Likelihood Estimation.
model.fit(train_data, estimator=MaximumLikelihoodEstimator)

print("\nLearned CPDs:")
for cpd in model.get_cpds():
    print(cpd)

# -----------------------------------------------------------
# Inference & Prediction using Variable Elimination
# -----------------------------------------------------------
# Create an inference object.
inference = VariableElimination(model)

# Prepare a list to store predictions
predictions = []

# Set your prediction threshold (0.5 in this case)
threshold = 0.5

# Loop over the test data instances to make predictions based on the threshold.
for index, row in test_data.drop(columns=["default_indicator"]).iterrows():
    evidence = row.to_dict()
    # Query the posterior distribution for 'default_indicator'
    posterior = inference.query(variables=['default_indicator'], evidence=evidence)
    # Get the probability of default_indicator for each state.
    # Assumption: The states are ordered such that index 1 corresponds to "default".
    prob_default = posterior.values[1]  # Adjust the index if your state order is different.
    # Apply threshold: if prob_default > threshold, classify as default (1)
    prediction = 1 if prob_default > threshold else 0
    predictions.append(prediction)

# Convert predictions to a pandas Series for evaluation purposes
predictions = pd.Series(predictions, index=test_data.index)

# Extract the ground truth from test_data.
y_true = test_data["default_indicator"]

# Evaluate predictions.
accuracy = accuracy_score(y_true, predictions)
print("\nClassification Accuracy:", accuracy)
print("\nClassification Report:")
print(classification_report(y_true, predictions))
print("Confusion Matrix:")
print(confusion_matrix(y_true, predictions))


Total data size: 183893


  0%|          | 0/1000000 [00:00<?, ?it/s]

Learned structure edges using K2 Score:
[('age_group', 'sub_grade'), ('age_group', 'issue_quarter'), ('age_group', 'purpose'), ('age_group', 'grade'), ('emp_length', 'age_group'), ('emp_length', 'issue_quarter'), ('emp_length', 'purpose'), ('emp_length', 'grade'), ('emp_length', 'home_ownership'), ('emp_length', 'fico_bucket_pct'), ('emp_length', 'sub_grade'), ('emp_length', 'int_rate_disc'), ('emp_length', 'loan_amnt_disc'), ('emp_length', 'open_acc_disc'), ('emp_length', 'total_acc_disc'), ('emp_length', 'revol_util_disc'), ('emp_length', 'default_indicator'), ('emp_length', 'inq_last_6mths_disc'), ('emp_length', 'revol_bal_disc'), ('emp_length', 'installment_disc'), ('emp_length', 'unemployment_rate_t1_disc'), ('emp_length', 'housing_price_t1_disc'), ('emp_length', 'fedfunds_t1_disc'), ('emp_length', 'delinq_2yrs_disc'), ('emp_length', 'gdp_growth_t1_disc'), ('emp_length', 'inflation_t1_disc'), ('emp_length', 'pub_rec_disc'), ('fico_bucket_pct', 'sub_grade'), ('fico_bucket_pct', 'is